# BBC NLP — Recommendation Engine
## BBC_NLP_PROJECT 2

---

### Overview
This notebook builds a **content-based recommendation engine** on top of the BBC NLP pipeline.
It loads the NER-enriched outputs from all 5 category notebooks and enables article search
and retrieval using a weighted multi-field scoring system.

---

### Data Sources
| Category      | Articles |
|---------------|----------|
| Business      | 510      |
| Sport         | 511      |
| Politics      | 417      |
| Tech          | 401      |
| Entertainment | 386      |
| **Total**     | **2225** |

---

### Dependencies
- `pandas` — data loading and manipulation
- `spacy (en_core_web_lg)` — NER and displacy rendering

## Phase 1 — Load & Combine All 5 Category Datasets
Loads all 5 NER-enriched CSV outputs, tags each with its category,
combines them into a single dataframe and generates a preview column.

In [1]:
# ============================================================
# PHASE 1: LOAD AND COMBINE ALL 5 CATEGORY DATASETS
# ============================================================
import pandas as pd

# Load each category NER output
business      = pd.read_csv('business_ner_output.csv')
entertainment = pd.read_csv('entertainment_ner_output.csv')
politics      = pd.read_csv('politics_ner_output.csv')
sport         = pd.read_csv('sport_ner_output.csv')
tech          = pd.read_csv('tech_ner_output.csv')

# Tag each with its category
business['category']      = 'Business'
entertainment['category'] = 'Entertainment'
politics['category']      = 'Politics'
sport['category']         = 'Sport'
tech['category']          = 'Tech'

# Combine into one dataframe
df = pd.concat([business, entertainment, politics, sport, tech], ignore_index=True)

# Add preview column (first 150 chars of cleaned_text)
df['preview'] = df['cleaned_text'].str[:150]

print(f" Total articles loaded: {len(df)}")
print(f" Categories: {df['category'].value_counts().to_dict()}")
print(f" Columns: {df.columns.tolist()}")

 Total articles loaded: 2225
 Categories: {'Sport': 511, 'Business': 510, 'Politics': 417, 'Tech': 401, 'Entertainment': 386}
 Columns: ['file_name', 'article_id', 'title', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'cleaned_text', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events', 'category', 'preview']


## Phase 2 — Recommendation Engine
Weighted multi-field search across all 2225 articles.
Returns the top N most relevant results with displacy NER highlighting.

### Search Fields & Weights
| Field                | Weight |
|----------------------|--------|
| named_persons        | 5      |
| person_roles         | 4      |
| named_organisations  | 3      |
| named_locations      | 2      |
| article_keywords     | 2      |
| title                | 2      |
| preview              | 1      |

In [2]:
# ============================================================
# PHASE 2: RECOMMENDATION ENGINE (PROFESSIONAL)
# ============================================================
import spacy

nlp = spacy.load('en_core_web_lg')

# ============================================================
# SEARCH FIELDS AND THEIR WEIGHTS
# ============================================================
SEARCH_FIELDS = {
    'named_persons':       5,
    'person_roles':        4,
    'named_organisations': 3,
    'named_locations':     2,
    'article_keywords':    2,
    'title':               2,
    'preview':             1,
}

# ============================================================
# SCORING FUNCTION
# ============================================================
def score_article(row, query):
    query_lower = query.lower()
    score = 0
    matched_fields = []

    for field, weight in SEARCH_FIELDS.items():
        field_value = str(row.get(field, '')).lower()
        if query_lower in field_value:
            score += weight
            matched_fields.append(field)

    return score, matched_fields

# ============================================================
# RECOMMEND FUNCTION
# ============================================================
def recommend(query, top_n=3):
    results = []

    for _, row in df.iterrows():
        score, matched_fields = score_article(row, query)
        if score > 0:
            results.append({
                'score':          score,
                'matched_fields': matched_fields,
                'category':       row['category'],
                'subcategory':    row['semantic_label'],
                'confidence':     row['confidence_score'],
                'title':          row['title'],
                'preview':        row['preview'],
                'cleaned_text':   row['cleaned_text'],
                'persons':        row['named_persons'],
                'orgs':           row['named_organisations'],
                'locations':      row['named_locations'],
            })

    # Sort by score descending
    results = sorted(results, key=lambda x: x['score'], reverse=True)

    if not results:
        print(f"  No articles found for: '{query}'")
        return

    # ============================================================
    # HEADER
    # ============================================================
    print("=" * 65)
    print(f"   BBC NLP — Recommendation Engine")
    print(f"   Search: \"{query}\"")
    print(f"   Found {len(results)} relevant articles (showing top {top_n})")
    print("=" * 65)

    for i, r in enumerate(results[:top_n], 1):

        # --------------------------------------------------------
        # RESULT BLOCK
        # --------------------------------------------------------
        print(f"\n  Result {i} of {top_n}")
        print(f"  {'─' * 60}")
        print(f"  Category:    {r['category']} — {r['subcategory']}")
        print(f"  Relevance:   {r['score']}  |  Matched in: {', '.join(r['matched_fields'])}")
        print(f"  Confidence:  {r['confidence']:.2f}")
        print(f"  {'─' * 60}")

        # NER fields
        if r['persons']:
            print(f"  People:    {r['persons']}")
        if r['orgs']:
            print(f"  Orgs:      {r['orgs']}")
        if r['locations']:
            print(f"  Locations: {r['locations']}")

        print(f"\n   {r['title']}")
        print(f"  {'─' * 60}")

        # displacy NER render
        doc = nlp(r['cleaned_text'][:2000])
        spacy.displacy.render(doc, style='ent', jupyter=True)

    # ============================================================
    # FOOTER
    # ============================================================
    print("\n" + "=" * 65)
    print(f"   Showing {min(top_n, len(results))} of {len(results)} results for \"{query}\"")
    print("=" * 65)

## Test Queries

In [3]:
recommend("Microsoft")

   BBC NLP — Recommendation Engine
   Search: "Microsoft"
   Found 84 relevant articles (showing top 3)

  Result 1 of 3
  ────────────────────────────────────────────────────────────
  Category:    Tech — Cybersecurity
  Relevance:   8  |  Matched in: named_organisations, article_keywords, title, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Stephen Toulouse | Windows Messenger | MSN Messenger
  Orgs:      Microsoft | Internet Explorer (IE) | Media Player | Sybari Software
  Locations: nan

   Microsoft releases bumper patches
  ────────────────────────────────────────────────────────────



  Result 2 of 3
  ────────────────────────────────────────────────────────────
  Category:    Tech — Cybersecurity
  Relevance:   8  |  Matched in: named_organisations, article_keywords, title, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Tony Macklin
  Orgs:      Microsoft | MSN | Google | Yahoo | Amazon | Blinkx
  Locations: nan

   Microsoft launches its own search
  ────────────────────────────────────────────────────────────



  Result 3 of 3
  ────────────────────────────────────────────────────────────
  Category:    Tech — Cybersecurity
  Relevance:   8  |  Matched in: named_organisations, article_keywords, title, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Stephen Toulouse | Windows Messenger | MSN Messenger
  Orgs:      Microsoft | Internet Explorer (IE) | Media Player | Sybari Software
  Locations: nan

   Microsoft releases patches
  ────────────────────────────────────────────────────────────



   Showing 3 of 84 results for "Microsoft"


In [4]:
recommend("Richard Wentwort")

   BBC NLP — Recommendation Engine
   Search: "Richard Wentwort"
   Found 1 relevant articles (showing top 3)

  Result 1 of 3
  ────────────────────────────────────────────────────────────
  Category:    Entertainment — Celebrity Sales
  Relevance:   9  |  Matched in: named_persons, person_roles
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Richard Wentworth | Tracey Emin | Henry Moore
  Orgs:      ArtWorks | Wentworth
  Locations: London | Tate Britain | Norway

   Gallery unveils interactive tree
  ────────────────────────────────────────────────────────────



   Showing 1 of 1 results for "Richard Wentwort"


In [5]:
recommend("Tony Blair")

   BBC NLP — Recommendation Engine
   Search: "Tony Blair"
   Found 143 relevant articles (showing top 3)

  Result 1 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Ministerial Scandals
  Relevance:   12  |  Matched in: named_persons, person_roles, article_keywords, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Tony Blair | Gerry Conlon - | Gerry Conlon | Anne Maguire | Daniel Day-Lewis
  Orgs:      Woolwich | the House of Commons | Guildford Four | the Court of Appeal | the Guildford Four
  Locations: UK | Guildford | Westminster | Ireland | London | Belfast

   PM apology over jailings
  ────────────────────────────────────────────────────────────



  Result 2 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Economic Policy
  Relevance:   12  |  Matched in: named_persons, person_roles, article_keywords, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Tony Blair | Carl Emmerson | David Page | Oliver Letwin
  Orgs:      pre-Budget | Labour | BBC Radio 4's | Napier University | the Institute for Fiscal Studies | BBC News | Investec Securities | IMF | the Institute of Fiscal Studies | Vincent Cable | the National Audit Office
  Locations: UK | Edinburgh | Britain

   Blair backs 'pre-election budget'
  ────────────────────────────────────────────────────────────



  Result 3 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Ministerial Scandals
  Relevance:   12  |  Matched in: named_persons, person_roles, article_keywords, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Tony Blair | David Blunkett | Kimberly Quinn | Michael Howard | Jack Straw | Philip Mawer | Alan Budd | Leoncia Casalme
  Orgs:      Home | Cabinet | the Home Office | the Commons Standards and Privileges Committee | House of Commons | High Court | the High Court
  Locations: Blunkett Sheffield | Sheffield | Westminster | UK

   Blair and Blunkett Sheffield trip
  ────────────────────────────────────────────────────────────



   Showing 3 of 143 results for "Tony Blair"


In [7]:
recommend("election")

   BBC NLP — Recommendation Engine
   Search: "election"
   Found 103 relevant articles (showing top 3)

  Result 1 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Electoral Reform
  Relevance:   6  |  Matched in: named_organisations, article_keywords, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Richard Mawrey | Muhammad Afzal | Mohammed Islam | Mohammed Kazi | Ravi Sukul
  Orgs:      an Election Court | Midlands Institute | Election Court | Aston
  Locations: Birmingham | QC | Birch Road East

   Labour trio 'had vote-rig factory'
  ────────────────────────────────────────────────────────────



  Result 2 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Conservative Party
  Relevance:   5  |  Matched in: article_keywords, title, preview
  Confidence:  1.00
  ────────────────────────────────────────────────────────────
  People:    Michael Howard | Tony Blair | Alan Milburn | Simon Hughes
  Orgs:      Labour | Whitehall | a Timetable for Action | BBC News | Thatcherism
  Locations: Northamptonshire | Britain | Iraq

   Howard unveils election platform
  ────────────────────────────────────────────────────────────



  Result 3 of 3
  ────────────────────────────────────────────────────────────
  Category:    Politics — Liberal Democrats
  Relevance:   5  |  Matched in: article_keywords, title, preview
  Confidence:  0.00
  ────────────────────────────────────────────────────────────
  People:    Sandy Walkington | Matthew Taylor
  Orgs:      Party | Lib Dem
  Locations: nan

   Lib Dems' new election PR chief
  ────────────────────────────────────────────────────────────



   Showing 3 of 103 results for "election"
